# AgentFix — run the real model in Google Colab

This notebook does **one thing**: it runs the finished agent against a real model, in the browser.

It exists for the `colab` tier — machines that cannot comfortably hold a local model. You still
do the whole workshop the normal way, in the IDE, on your own machine:

- read every lesson,
- write the three pieces of the agent yourself (the `run_tests` tool and its schema, the loop's
  tool dispatch, the verification-based stop condition),
- run the exercise tests locally — they use a scripted fake model, so they need no model at all
  and they pass on any machine.

The only step that needs a real model is the last one: watching the agent actually fix a bug.
That step happens here.

**Do not run `python run.py doctor` on your own machine.** It will fail, because there is no
Ollama and no model there — that is expected and fine on this tier. The doctor check that
matters for you runs in this notebook, below.

**The code you run here is the reference solution, not your own edits.** This notebook checks out
the finished versions of the two exercise files so the agent is guaranteed to work. Your own
implementation stays on your machine, verified by the exercise tests you already ran there.

## 0. Turn on the GPU

**Runtime → Change runtime type → T4 GPU**, then run the cells below in order, top to bottom.

CPU also works. It is just slower.

In [ ]:
!python --version
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "No GPU — CPU inference will work, just slower."

## 1. Configuration

Colab runs `qwen3:1.7b`, the same fallback the local Option 2 tier uses. It is a thinking
model: each turn reasons before it acts, so turns take longer than they would on Mellum2.

`MELLUM_MODEL` is the environment variable AgentFix reads for the Ollama model name.

In [ ]:
REPO_URL = "https://github.com/jelenadjuric01/agentfix-workshop.git"
CHECKOUT = "/content/agentfix-workshop"

BASE_MODEL = "qwen3:1.7b"
WORKSHOP_MODEL = "agentfix-qwen3"
CONTEXT_LENGTH = 16384

print("Repository:", REPO_URL)
print("Checkout:  ", CHECKOUT)
print("Model:     ", BASE_MODEL, "->", WORKSHOP_MODEL)
print("Context:   ", CONTEXT_LENGTH)

## 2. Install and start Ollama

Colab may not include `zstd`, which the current Ollama Linux installer needs.

The server is started in the background and stays available to every later cell.

In [ ]:
!apt-get update -qq
!apt-get install -y -qq zstd pciutils
!if ! command -v ollama >/dev/null 2>&1; then curl -fsSL https://ollama.com/install.sh | sh; fi
!ollama --version

In [ ]:
%env OLLAMA_CONTEXT_LENGTH=16384

!if ! curl -fsS http://127.0.0.1:11434/api/version >/dev/null 2>&1; then nohup ollama serve > /tmp/ollama-colab.log 2>&1 & sleep 4; fi
!curl -fsS http://127.0.0.1:11434/api/version

## 3. Pull the base model and derive the workshop model

Do not skip the `ollama create` step. It is what gives the model the 16,384-token context window
the agent needs — without it Ollama quietly uses 4,096 and the agent loses its own system prompt
partway through a long run.

In [ ]:
!ollama pull qwen3:1.7b
!printf "FROM qwen3:1.7b\nPARAMETER num_ctx 16384\n" > /tmp/Modelfile.agentfix-qwen3
!ollama create agentfix-qwen3 -f /tmp/Modelfile.agentfix-qwen3

%env MELLUM_MODEL=agentfix-qwen3

!ollama list

In [ ]:
# Quick smoke test — the model should reply with exactly READY.
!ollama run agentfix-qwen3 "Reply with exactly: READY"

## 4. Clone the repository and check out the finished agent

This clones `main` and then checks out the two exercise files from the `stage-3-solution` tag, so
the agent in this runtime is complete and will run.

Reading the solution here does not replace the exercises — you already did those on your machine
against the fake model.

In [ ]:
%cd /content

!rm -rf /content/agentfix-workshop
!git clone --branch main https://github.com/jelenadjuric01/agentfix-workshop.git /content/agentfix-workshop

%cd /content/agentfix-workshop

!git fetch --all --tags --prune
!git checkout stage-3-solution -- src/agentfix/tools/tests_tool.py src/agentfix/agent/loop.py

!echo
!echo "Finished agent files in place — no TODO markers should be listed below:"
!grep -n "TODO(stage-" src/agentfix/tools/tests_tool.py src/agentfix/agent/loop.py || echo "none"

## 5. Install AgentFix

In [ ]:
!python -m pip install -q -e ".[dev]"

## 6. Check the setup

This is the `doctor` check — the one you skip on your own machine. Everything should read
`[PASS]` except RAM, which is fine in Colab.

The line to look at is **`context window: 16384`**. If it says `4096`, rerun the `ollama create`
cell in section 3.

In [ ]:
!agentfix doctor

## 7. Run the agent against a real model

The first task is a straightforward bug.

In [ ]:
!agentfix solve tasks/workshop/01-shopcart --verbose

Now the harder one, where the bug is **not** in the file the failing test points at — which is
why `list_files` and `read_file` earn their place:

In [ ]:
!agentfix solve tasks/workshop/02-invoice --verbose

`--verbose` prints the trace built by the tool dispatch you wrote in Stage 2. Read it. You should
see the model call `run_tests`, look around, write a file, and run the tests again. That last call
is what ends the run, because of the stop condition from Stage 3.

If a run burns all ten steps and prints `NOT SOLVED`, that is not necessarily a bug. Real models
do not fix every task, and `qwen3:1.7b` is much smaller than Mellum2 and may not do as well.
Read the trace rather than the verdict. Rerun it, or move on.

## 8. Optional — the evaluation suite

Runs the model repeatedly, so it takes a while.

In [ ]:
!agentfix eval --suite workshop --limit 3